# 09 — Agents

**Module notebook — definitions only.**

Wraps the existing pipeline (01-04, 07) and the LlamaIndex-backed RAG retriever
(08) into two agent functions that operate on a shared state dict, so they can
be plugged into the LangGraph orchestrator in `11_orchestrator.ipynb` unchanged.

Each agent takes the current state, does its job, and returns an **updated copy**
of the state (never mutates in place) — this is what makes them safe to use as
LangGraph nodes.

Depends on: `01_audio_processor`, `02_transcriber`, `03_summarizer`, `04_extractor`,
`06_rag_engine` (for `RAG_SYSTEM_PROMPT_TEMPLATE`), `07_report_generator`,
`08_llamaindex_retriever` (all loaded earlier in `main.ipynb`).

## Content agent

Runs the full ingest -> transcript -> translate -> summary -> extraction -> PDF
pipeline. This is what should run for a brand-new recording.

Takes **three independent settings** from state:
- `audio_type` (`"auto"` / `"hinglish"`) — which ASR model transcribes the audio.
- `transcript_language` (`"english"` / `"arabic"`) — what language the
  "Translated Transcript" tab is written in.
- `summary_language` (`"english"` / `"arabic"`) — what language the summary,
  action items, key decisions, open questions, and PDF report are written in.

These used to be a single `language` setting doing all three jobs — now
`audio_type` no longer forces a transcription language at all (Whisper always
auto-detects; see `02_transcriber.ipynb`), and `transcript_language` /
`summary_language` can differ from each other and from the audio's actual
spoken language.

Handles two edge cases directly, rather than letting them crash the graph:
- **Broken/private/unavailable YouTube URL, or an unsupported/corrupted local file** —
  `process_input`/`transcribe_all` are wrapped in `try/except`; any failure becomes a
  clean `blocked` state with a readable message instead of an unhandled traceback.
- **Silent or near-silent audio** — if transcription comes back empty or
  near-empty, we stop before wasting LLM calls translating/summarizing
  nothing, and report it clearly instead of returning a hallucinated-looking
  empty summary.

In [ ]:
import os
import tempfile
import uuid

MIN_TRANSCRIPT_CHARS = 20  # below this, treat the transcript as empty/unusable


def content_agent(state: dict) -> dict:
    """Process a new source end-to-end: raw transcript, translated transcript,
    title, summary, extraction, PDF."""
    source = state["source"]
    audio_type = state.get("audio_type", "auto")
    transcript_language = state.get("transcript_language", "english")
    summary_language = state.get("summary_language", "english")
    errors = list(state.get("errors", []))

    print("content_agent: starting pipeline for", source)

    try:
        chunks = process_input(source)
        raw_transcript = transcribe_all(chunks, audio_type=audio_type)
    except Exception as e:
        # Covers: broken/private/unavailable YouTube URL, and unsupported/corrupted
        # local audio files — both raise from 01_audio_processor, caught here so
        # the graph run ends cleanly instead of crashing the kernel.
        errors.append(f"Could not process the audio source: {e}")
        print(f"content_agent: {errors[-1]}")
        return {**state, "errors": errors, "blocked": True}

    print(f"content_agent: raw transcript ready ({len(raw_transcript)} characters).")

    if len(raw_transcript.strip()) < MIN_TRANSCRIPT_CHARS:
        # Covers: silent/near-silent audio — transcription returns an empty or
        # near-empty transcript. Stop here rather than translating/summarizing
        # nothing.
        errors.append(
            "The audio appears to be silent, near-silent, or unintelligible — "
            "the transcript came back empty or too short to summarize. Check "
            "the recording and try again."
        )
        print(f"content_agent: {errors[-1]}")
        return {
            **state,
            "raw_transcript": raw_transcript,
            "errors": errors,
            "blocked": True,
        }

    print(f"content_agent: translating transcript to '{transcript_language}'...")
    transcript = translate_transcript(raw_transcript, language=transcript_language)

    title = generate_title(transcript, language=summary_language)
    summary = summarize(transcript, language=summary_language)
    action_items = extract_action_items(transcript, language=summary_language)
    key_decisions = extract_key_decisions(transcript, language=summary_language)
    open_questions = extract_questions(transcript, language=summary_language)

    report_data = {
        "title": title,
        "summary": summary,
        "action_items": action_items,
        "key_decisions": key_decisions,
        "open_questions": open_questions,
    }
    # Fix: a fixed relative filename here means two concurrent runs
    # (two browser sessions, or two quick clicks) race on the exact same
    # "meeting_report.pdf" in the working directory - one session can read
    # back a mix of both PDFs, or hit a "file in use" error on Windows,
    # the same class of bug the streamlit_app.py comments describe as
    # already fixed for the transcript-tab PDFs. Give the main report the
    # same treatment: a unique temp path per run.
    pdf_path = generate_pdf_report(
        report_data,
        output_path=os.path.join(tempfile.gettempdir(), f"meeting_report_{uuid.uuid4().hex}.pdf"),
        language=summary_language,
    )
    print(f"content_agent: PDF report saved to {pdf_path}")

    return {
        **state,
        "raw_transcript": raw_transcript,
        "transcript": transcript,
        "title": title,
        "summary": summary,
        "action_items": action_items,
        "key_decisions": key_decisions,
        "open_questions": open_questions,
        "pdf_path": pdf_path,
        "errors": errors,
    }


## RAG agent

Answers a question against an already-processed transcript. Builds the LlamaIndex
retriever once (lazily, cached in state as `llama_index_bundle`) and reuses it for
every follow-up question, instead of rebuilding the index on every call.

Indexes **`raw_transcript`** (the original words, auto-detected language/script)
rather than the translated transcript — so chat answers come from exactly what
was said, not a translated copy, regardless of what `transcript_language` was
chosen for the display tab. `ask_llama_question` (`08_llamaindex_retriever.ipynb`)
separately detects the *question's* own language and answers in kind.

The **no-relevant-chunk edge case** (a question genuinely not covered by the
transcript) is handled inside `ask_llama_question` via a relevance-score
threshold, not here.

In [ ]:
def rag_agent(state: dict) -> dict:
    """Answer state["question"] against state["raw_transcript"], using
    LlamaIndex for retrieval (08_llamaindex_retriever.ipynb). Requires
    content_agent to have run first (or a raw_transcript to already be
    present in state)."""
    raw_transcript = state.get("raw_transcript")
    if not raw_transcript:
        errors = state.get("errors", []) + [
            "rag_agent: no transcript in state yet — run the content agent first."
        ]
        print(errors[-1])
        return {**state, "answer": None, "errors": errors}

    summary_language = state.get("summary_language", "english")
    question = state.get("question")

    index_bundle = state.get("llama_index_bundle")
    if index_bundle is None:
        print("rag_agent: building LlamaIndex retriever for this transcript (first question).")
        index_bundle = build_llama_index_bundle(raw_transcript)

    answer = ask_llama_question(index_bundle, question, language=summary_language) if question else None

    chat_history = state.get("chat_history", [])
    if question:
        chat_history = chat_history + [{"question": question, "answer": answer}]

    return {**state, "llama_index_bundle": index_bundle, "answer": answer, "chat_history": chat_history}
